# Free Colab: ForTy GPU Pipeline Check

Upload this notebook at https://colab.research.google.com and sign in to your
Google account. Select an available GPU under **Runtime > Change runtime type**,
then run the cells in order. Free GPU availability, quotas and session duration
vary. This notebook does not provision paid VMs or activate a paid plan.

The default uses the full model and trains ComboLoss and plain cross-entropy
for **two epochs each**, with one shard per split, one seed and batch size 1.
This is a small real-source workflow check. Scores are diagnostic and do not
establish a model ranking. Do not tune the model or training settings using
test metrics.

Have the supplied `publication-recovery.zip` or
`geospatial-mamba-deforestation.bundle` ready for the upload cell.
The notebook pins the source to a verified commit and does not depend on
unpublished GitHub changes. This entry point has not been executed on an actual
Colab GPU; the runtime checks below determine whether the session is usable.

In [ ]:
import datetime
import io
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import uuid
import zipfile

SOURCE_COMMIT = "ae6a123d0469a2950ce6b9733cf7902b14cb7f5c"
SOURCE_BRANCH = "codex/reproducible-evaluation-performance"
DATA_ROOT = "gs://forest_typology/forty_v1/1.0.0"
TRAIN_SHARDS = 1
VAL_SHARDS = 1
TEST_SHARDS = 1
EPOCHS = 2
BATCH_SIZE = 1
SEEDS = [42]
SAVE_TO_DRIVE = False

WORKSPACE = Path("/content/geospatial-mamba")
REPO = WORKSPACE / "source"
WORKSPACE.mkdir(parents=True, exist_ok=True)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_base = Path("/content/drive/MyDrive/geospatial-mamba")
else:
    output_base = WORKSPACE / "artifacts"
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SESSION_ROOT = output_base / ("colab-" + stamp + "-" + uuid.uuid4().hex[:8])
SESSION_ROOT.mkdir(parents=True, exist_ok=False)
RUN_ROOT = SESSION_ROOT / "experiment"
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE / "cache/matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(WORKSPACE / "cache"))
print("Artifacts:", SESSION_ROOT)
print("Each arm:", EPOCHS, "epochs; train/val/test shards:",
      TRAIN_SHARDS, VAL_SHARDS, TEST_SHARDS)

Set `SAVE_TO_DRIVE=True` to authorize mounting your Google Drive and write
outputs there from the start. The default `/content` directory is temporary;
download results promptly when the run finishes. Forced runtime reclamation
may prevent export. Drive preserves only files that were successfully written.

Training does not resume optimizer state. After an interruption, rerun in a
fresh directory. Rerunning the configuration cell creates a new experiment
directory and preserves earlier directories.

In [ ]:
def bundle_bytes(filename, payload):
    if filename.endswith(".bundle"):
        return payload
    if not filename.endswith(".zip"):
        raise ValueError("Select the supplied .zip recovery archive or .bundle file")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        members = [item for item in archive.infolist()
                   if item.filename == "geospatial-mamba-deforestation.bundle"]
        if len(members) != 1:
            raise ValueError("The recovery archive must contain exactly one project Git bundle")
        if members[0].file_size > 64 * 1024 * 1024:
            raise ValueError("The Git bundle exceeds the size limit for this notebook")
        return archive.read(members[0])

if not REPO.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one recovery archive at a time")
    filename, payload = next(iter(uploaded.items()))
    bundle = WORKSPACE / ("source-" + uuid.uuid4().hex + ".bundle")
    bundle.write_bytes(bundle_bytes(filename, payload))
    subprocess.run(["git", "clone", "--branch", SOURCE_BRANCH,
                    str(bundle), str(REPO)], check=True)
    subprocess.run(["git", "bundle", "verify", str(bundle)], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT],
                   cwd=REPO, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"],
                               cwd=REPO, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"],
                                cwd=REPO, text=True)
if head != SOURCE_COMMIT or dirty:
    raise RuntimeError("Source commit mismatch or modified working tree; check the source and use a fresh workspace")
(SESSION_ROOT / "source_commit.txt").write_text(head + "\n")
print("Source:", head)

In [ ]:
# Reuse the TensorFlow/CUDA stack provided by the Colab runtime.
probe_code = (
    "import json, tensorflow as tf; "
    "print(json.dumps({'version': tf.__version__, "
    "'schemes': tf.io.gfile.get_registered_schemes()}))"
)
probe = subprocess.check_output([sys.executable, "-c", probe_code], text=True)
runtime = json.loads(probe.strip().splitlines()[-1])
major_minor = tuple(int(part) for part in runtime["version"].split(".")[:2])
if major_minor < (2, 15):
    raise RuntimeError("This notebook requires a Colab runtime with TensorFlow 2.15 or newer")
if "gs" not in runtime["schemes"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps",
                    "tensorflow-io-gcs-filesystem"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO)],
               check=True)
packages = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(SESSION_ROOT / "packages.txt").write_text(packages)
print("Colab TensorFlow:", runtime["version"])

In [ ]:
# Require real GPU convolution gradients and GCS registration before data IO.
subprocess.run([sys.executable, str(REPO / "cloud/check_runtime.py"),
                "--output", str(SESSION_ROOT / "runtime-readiness.json")],
               cwd=REPO, check=True)
readiness = json.loads((SESSION_ROOT / "runtime-readiness.json").read_text())
print(json.dumps(readiness, indent=2))

The following cells generate and display the plan before execution.
Preflight reads one example from each train/validation/test split to check
format; this does not validate the full dataset or geographic independence.
Test metrics are computed only after both training arms finish.

Freeze the settings here. These small-run test scores must not select
hyperparameters for a later formal experiment. Diagnostics are saved in the
session directory; the runner uses a fresh `experiment` subdirectory.

In [ ]:
def experiment_command(execute=False):
    command = [sys.executable, "-m", "mamba_forest.experiment",
               "--run-root", str(RUN_ROOT), "--data-root", DATA_ROOT,
               "--train-shards", str(TRAIN_SHARDS), "--val-shards", str(VAL_SHARDS),
               "--test-shards", str(TEST_SHARDS), "--epochs", str(EPOCHS),
               "--batch-size", str(BATCH_SIZE), "--seeds", *map(str, SEEDS)]
    if execute:
        command.append("--execute")
    return command

subprocess.run(experiment_command(), cwd=REPO, check=True)
plan = json.loads((RUN_ROOT / "plan.json").read_text())
print(json.dumps(plan["config"], indent=2))
print("Stages:", [step["id"] for step in plan["steps"]])

In [ ]:
def archive_session():
    destination = SESSION_ROOT.parent / (SESSION_ROOT.name + ".zip")
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(SESSION_ROOT.rglob("*")):
            if path.is_file() and not path.is_symlink():
                archive.write(path, path.relative_to(SESSION_ROOT))
    print("Archive:", destination)
    return destination

def execute_with_cleanup(argv, cwd):
    process = subprocess.Popen(argv, cwd=cwd, start_new_session=True)
    try:
        return_code = process.wait()
    except BaseException:
        try:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
        finally:
            # A stopped runner can leave a child in its process group.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
        raise
    if return_code:
        raise subprocess.CalledProcessError(return_code, argv)

In [ ]:
try:
    execute_with_cleanup(experiment_command(execute=True), cwd=REPO)
    report = json.loads((RUN_ROOT / "comparison.json").read_text())
    print("Configured data source:", DATA_ROOT)
    print("Small-source pipeline scores only; no real-data model ranking is established.")
    print(json.dumps(report["summary"], indent=2))
finally:
    ARCHIVE = archive_session()

After success or an ordinary error, the notebook attempts to archive
weights, logs, configuration and metrics. Stopping a cell first cleans up the
training processes. Archiving cannot be guaranteed if the entire runtime is
reclaimed. After resolving a failure, use a fresh experiment directory.

The cell below can repackage and download existing files. With Drive enabled,
outputs are also available in your Drive. See `docs/gpu-validation.md` in the
repository for the formal three-seed comparison and full-test evaluation;
free quotas may not be sufficient.

In [ ]:
from google.colab import files
ARCHIVE = archive_session()
files.download(str(ARCHIVE))